In [0]:
import json

def load_config(config_path):
    """Load ETL configuration from a JSON file."""
    with open(config_path, "r") as f:
        config = json.load(f)
    print(f"✔ Config loaded from: {config_path}")
    return config


def configure_adls_oauth(storage_account, client_id, client_secret, tenant_id):
    """Configure OAuth (Service Principal) authentication for ADLS Gen2."""
    base = f"{storage_account}.dfs.core.windows.net"
    endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"

    spark.conf.set(f"fs.azure.account.auth.type.{base}", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{base}",
                   "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{base}", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{base}", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{base}", endpoint)
    print(f"✔ OAuth configured for: {storage_account}")


def get_adls_path(container, storage_account, path=""):
    """Build an ABFSS URI for ADLS Gen2."""
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base}/{path}" if path else f"{base}/"


def list_adls_files(container, storage_account, path=""):
    """List files/folders at an ADLS Gen2 path."""
    full_path = get_adls_path(container, storage_account, path)
    return dbutils.fs.ls(full_path)


def read_adls_file(container, storage_account, file_path, file_format="csv", **options):
    """Read a file from ADLS Gen2 into a DataFrame."""
    full_path = get_adls_path(container, storage_account, file_path)
    reader = spark.read.format(file_format)
    for key, value in options.items():
        reader = reader.option(key, value)
    return reader.load(full_path)


def write_adls_file(df, container, storage_account, output_path, file_format="parquet", mode="overwrite"):
    """Write a DataFrame to ADLS Gen2."""
    full_path = get_adls_path(container, storage_account, output_path)
    df.write.format(file_format).mode(mode).save(full_path)
    print(f"✔ Data written to: {full_path}")

In [0]:
# --- Load configuration from etl_config.json ---
CONFIG_PATH = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/etl_config.json"
config = load_config(CONFIG_PATH)

# Extract connection settings
adls_cfg   = config["adls_connection"]
file_cfg   = config["file_paths"]
output_cfg = config["output"]

STORAGE_ACCOUNT = adls_cfg["storage_account"]
CONTAINER       = adls_cfg["container"]
CLIENT_ID       = adls_cfg["client_id"]
TENANT_ID       = adls_cfg["tenant_id"]

# Fetch client secret securely from Databricks Secret Scope
CLIENT_SECRET = dbutils.secrets.get(scope=adls_cfg["secret_scope"], key=adls_cfg["secret_key"])

configure_adls_oauth(STORAGE_ACCOUNT, CLIENT_ID, CLIENT_SECRET, TENANT_ID)

In [0]:
# List root of container
display(list_adls_files(CONTAINER, STORAGE_ACCOUNT))

# List files in ainput1 subfolder
display(list_adls_files(CONTAINER, STORAGE_ACCOUNT, "ainput1"))

In [0]:
df_csv = read_adls_file(CONTAINER, STORAGE_ACCOUNT, file_cfg["csv"], "csv", header="true")
display(df_csv)

In [0]:
df_json = read_adls_file(CONTAINER, STORAGE_ACCOUNT, file_cfg["json"], "json")
display(df_json)

In [0]:
df_parquet = read_adls_file(CONTAINER, STORAGE_ACCOUNT, file_cfg["parquet"], "parquet")
display(df_parquet)

In [0]:
write_adls_file(df_parquet, CONTAINER, STORAGE_ACCOUNT, output_cfg["path"], output_cfg["format"], output_cfg["mode"])

In [0]:
# Alternate CSV read using config - demonstrates reusability
df_csv_alt = read_adls_file(CONTAINER, STORAGE_ACCOUNT, file_cfg["csv"], "csv", header="true")
display(df_csv_alt)

In [0]:
# List all available secret scopes
scopes = dbutils.secrets.listScopes()
display(scopes)

# Uncomment below to list secrets within a specific scope
# display(dbutils.secrets.list("your-scope-name"))

In [0]:
# Verify config loaded correctly (secrets are redacted by Databricks)
print("=== ETL Config Summary ===")
print(f"Storage Account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"Client ID       : {CLIENT_ID}")
print(f"Tenant ID       : {TENANT_ID}")
print(f"Secret Scope    : {adls_cfg['secret_scope']}")
print(f"Secret Key      : {adls_cfg['secret_key']}")
print(f"CSV Path        : {file_cfg['csv']}")
print(f"JSON Path       : {file_cfg['json']}")
print(f"Parquet Path    : {file_cfg['parquet']}")
print(f"Output Path     : {output_cfg['path']}")
print(f"Output Format   : {output_cfg['format']}")
print(f"Output Mode     : {output_cfg['mode']}")